# Experimento 01: Enpoint online administrado para VLM

**Objetivo**: Implementar Qwen2.5-VL-32B-Instruct como un endpoint online administrado de Azure ML
a través de la colección Hugging Face en Microsoft Foundry.

**Costo**: ~$0.90 (ejecutando A100 durante ~15 minutos)

## Lo que demuestra este cuaderno
- La cuota de GPU funciona para `Standard_NC24ads_A100_v4`
- Los modelos del registro de Hugging Face están disponibles en nuestra región
- La inferencia VLM multiimagen funciona a través de una API compatible con OpenAI
- Podemos medir la latencia real, el costo y el comportamiento de arranque en frío

## 1. Setup y Autenticacion

In [3]:
# Install dependencies (uncomment if needed)
# !pip install azure-ai-ml azure-identity openai python-dotenv --upgrade --quiet

In [5]:
import os
import sys
import json
import time
from uuid import uuid4
from dotenv import load_dotenv

load_dotenv()

from azure.ai.ml import MLClient
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment
from azure.identity import DefaultAzureCredential
from openai import OpenAI

print("All imports OK")

All imports OK


In [6]:
# Variables de entorno para la conexión a Azure
SUBSCRIPTION_ID = os.getenv("SUBSCRIPTION_ID", "<your-subscription-id>")
RESOURCE_GROUP = os.getenv("RESOURCE_GROUP", "<your-resource-group>")
WORKSPACE_NAME = os.getenv("WORKSPACE_NAME", "<your-workspace-name>")
LOCATION = os.getenv("LOCATION", "eastus")

# Endpoint names (must be globally unique per region)
ENDPOINT_NAME = f"bogota-online-{str(uuid4())[:8]}"
DEPLOYMENT_NAME = f"qwen-vl-{str(uuid4())[:8]}"

print(f"Subscription: {SUBSCRIPTION_ID}")
print(f"Workspace:    {WORKSPACE_NAME}")
print(f"Region:       {LOCATION}")
print(f"Endpoint:     {ENDPOINT_NAME}")
print(f"Deployment:   {DEPLOYMENT_NAME}")

Subscription: <your-subscription-id>
Workspace:    <your-workspace-name>
Region:       eastus
Endpoint:     bogota-online-3650bdb1
Deployment:   qwen-vl-e71de483


In [ ]:
# Authenticate
credential = DefaultAzureCredential()

client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)

# Verify workspace
ws = client.workspaces.get(WORKSPACE_NAME)
print(f"✅ Connected to workspace: {ws.name} ({ws.location})")
print(f"   Description: {ws.description or 'N/A'}")

## 2. Build Model URI

The HuggingFace collection on Azure uses a specific URI format:
`azureml://registries/HuggingFace/models/<model-id-on-azure>/labels/latest`

In [ ]:
# Model: Qwen2.5-VL-32B-Instruct
MODEL_NAME = "Qwen/Qwen2.5-VL-32B-Instruct"

# Convert to Azure ML registry format
model_uri = (
    f"azureml://registries/HuggingFace/models/"
    f"{MODEL_NAME.replace('/', '-').replace('_', '-').lower()}/labels/latest"
)

print(f"Model URI: {model_uri}")

# Expected: azureml://registries/HuggingFace/models/qwen-qwen2.5-vl-32b-instruct/labels/latest

## 3. Deploy Managed Online Endpoint

This creates the endpoint AND the deployment. The deployment uses your A100-80GB GPU.

⏱️ **This cell takes 10-20 minutes to complete.**

In [ ]:
# Your GPU instance type
INSTANCE_TYPE = "Standard_NC24ads_A100_v4"

# Create endpoint
print(f"Creating endpoint: {ENDPOINT_NAME}...")
endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="Bogota land-use VLM — experimental online endpoint",
    auth_mode="key",
)

deployment_start = time.time()

try:
    client.begin_create_or_update(endpoint).wait()
    print(f"✅ Endpoint created")
except Exception as e:
    print(f"❌ Endpoint creation failed: {e}")
    print("\nCheck:")
    print("  1. Quota for Standard_NC24ads_A100_v4 in your region")
    print("  2. Resource group and workspace names")
    print("  3. Azure CLI is logged in (az login)")
    raise

# Create deployment
print(f"\nCreating deployment: {DEPLOYMENT_NAME}...")
print(f"  Instance type: {INSTANCE_TYPE}")
print(f"  Instance count: 1")

deployment = ManagedOnlineDeployment(
    name=DEPLOYMENT_NAME,
    endpoint_name=ENDPOINT_NAME,
    model=model_uri,
    instance_type=INSTANCE_TYPE,
    instance_count=1,
    request_settings={
        "max_concurrent_requests_per_instance": 1,
        "request_timeout_ms": 120000,
    },
)

try:
    client.online_deployments.begin_create_or_update(deployment).wait()
    deployment_elapsed = time.time() - deployment_start
    print(f"✅ Deployment complete in {deployment_elapsed/60:.1f} minutes")
except Exception as e:
    print(f"❌ Deployment failed: {e}")
    print("\nCheck:")
    print("  1. GPU quota: check Azure Portal → Machine Learning → Quota")
    print("  2. Model availability: check https://ml.azure.com → Model catalog")
    print("  3. Project is Hub-based (required for HuggingFace collection)")
    raise

# Set 100% traffic to the deployment
endpoint.traffic = {DEPLOYMENT_NAME: 100}
client.begin_create_or_update(endpoint).wait()
print(f"✅ Traffic set to 100% → {DEPLOYMENT_NAME}")

## 4. Get Endpoint Credentials

In [ ]:
# Get API key
keys = client.online_endpoints.get_keys(ENDPOINT_NAME)
api_key = keys.primary_key

# Build API URL
api_url = f"https://{ENDPOINT_NAME}.{LOCATION}.inference.ml.azure.com/v1"

# Get scoring URI for reference
ep = client.online_endpoints.get(ENDPOINT_NAME)
scoring_uri = ep.scoring_uri

print(f"API URL:   {api_url}")
print(f"Scoring:   {scoring_uri}")
print(f"API Key:   {api_key[:12]}...{api_key[-4:]}")

# Save to .env for later use
with open("../../.env", "a") as f:
    f.write(f"\n# Experimental online endpoint\n")
    f.write(f"EXP_ONLINE_ENDPOINT={ENDPOINT_NAME}\n")
    f.write(f"EXP_ONLINE_DEPLOYMENT={DEPLOYMENT_NAME}\n")
    f.write(f"EXP_ONLINE_API_KEY={api_key}\n")
    f.write(f"EXP_ONLINE_API_URL={api_url}\n")
print("✅ Credentials saved to .env")

## 5. Test: Single Image Inference

In [ ]:
# Create OpenAI-compatible client
openai_client = OpenAI(
    base_url=f"{api_url}/v1",
    api_key=api_key,
    default_headers={"azureml-model-deployment": DEPLOYMENT_NAME},
)

# Test image: Flatiron Building (public domain)
test_image = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4f/Flatiron_Building_3618433845_5745e68717.jpg/800px-Flatiron_Building_3618433845_5745e68717.jpg"

print("Sending single-image inference request...")
start = time.time()

response = openai_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe this building in one sentence. Is it residential, commercial, or mixed-use?"},
                {"type": "image_url", "image_url": {"url": test_image}},
            ],
        }
    ],
    max_tokens=128,
)

single_latency = time.time() - start
print(f"\n✅ Response received in {single_latency:.1f}s")
print(f"\nModel says:")
print(response.choices[0].message.content)
print(f"\nToken usage: {response.usage}")

# Record measurement
SINGLE_IMAGE_LATENCY = single_latency

## 6. Test: Multi-Image Inference (4 images = 1 building)

In [ ]:
# Simulate a building with 4 views
building_images = [
    "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4f/Flatiron_Building_3618433845_5745e68717.jpg/800px-Flatiron_Building_3618433845_5745e68717.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/5/5e/Upper_East_Side_apartment_buildings.jpg/800px-Upper_East_Side_apartment_buildings.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/8/8b/Brownstones_at_112-114_East_74th_Street.jpg/800px-Brownstones_at_112-114_East_74th_Street.jpg",
]

print(f"Sending multi-image request with {len(building_images)} images...")
start = time.time()

# Build content with all images
content = [
    {
        "type": "text",
        "text": (
            "You are looking at multiple views of buildings in Bogota, Colombia. "
            "Classify the PRIMARY use type of the building shown. "
            "Return a JSON object with: "
            '"label": one of [RESIDENCIAL_1, COMERCIAL_1, COMERCIAL_2, COMERCIAL_3, DOTACIONAL_1, DOTACIONAL_2, MOLES_1, RURAL_1, MIXTO_1, MIXTO_2, UNKNOWN], '
            '"confidence": 0-1, '
            '"evidence": ["visible feature 1", "visible feature 2"]'
        ),
    }
]

for url in building_images:
    content.append({"type": "image_url", "image_url": {"url": url}})

response = openai_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": content}],
    max_tokens=256,
)

multi_latency = time.time() - start
print(f"\n✅ Response received in {multi_latency:.1f}s")
print(f"\nRaw response:")
print(response.choices[0].message.content[:400])

# Try to parse JSON
try:
    parsed = json.loads(response.choices[0].message.content)
    print(f"\n✅ JSON parsed successfully!")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError:
    print(f"\n⚠ JSON parse failed — this is expected without schema enforcement (Phase 03 fixes this)")

MULTI_IMAGE_LATENCY = multi_latency

## 7. Measure Cold Start (Optional)

In [ ]:
# Skip this cell if you're running a quick test.
# Cold start measurement takes ~5 minutes.

MEASURE_COLD_START = False  # Set to True to measure

if MEASURE_COLD_START:
    print("Scaling endpoint to 0...")
    
    # Scale to 0 (stop billing)
    endpoint.traffic = {}
    client.begin_create_or_update(endpoint).wait()
    print("Waiting 60 seconds for deallocation...")
    time.sleep(60)
    
    # Scale back up
    print("Scaling endpoint back to 100%...")
    endpoint.traffic = {DEPLOYMENT_NAME: 100}
    scale_start = time.time()
    client.begin_create_or_update(endpoint).wait()
    scale_time = time.time() - scale_start
    print(f"Scale-up API call: {scale_time:.0f}s")
    
    # Wait for endpoint to be ready, then test
    print("Waiting for model to load (up to 120s)...")
    for i in range(12):
        time.sleep(10)
        try:
            r = openai_client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": "ping"}],
                max_tokens=5,
            )
            cold_start_total = time.time() - scale_start
            print(f"✅ Endpoint ready after {cold_start_total:.0f}s (cold start)")
            COLD_START_TIME = cold_start_total
            break
        except Exception:
            print(f"   Still loading... ({i+1}0s)")
else:
    print("Skipping cold start measurement (set MEASURE_COLD_START=True to run)")
    COLD_START_TIME = "not measured"

## 8. Latency & Cost Summary

In [ ]:
A100_COST_PER_HOUR = 3.60  # USD
A100_COST_PER_SEC = A100_COST_PER_HOUR / 3600

print("=" * 50)
print("ONLINE ENDPOINT RESULTS")
print("=" * 50)
print(f"Model:            {MODEL_NAME}")
print(f"Instance:         {INSTANCE_TYPE}")
print(f"Deployment time:  {deployment_elapsed/60:.1f} min")
print(f"Single img lat:   {SINGLE_IMAGE_LATENCY:.1f}s")
print(f"Multi img ({len(building_images)}):  {MULTI_IMAGE_LATENCY:.1f}s")
print(f"Cold start:       {COLD_START_TIME}")
print(f"")
print(f"Cost per call (single): ${SINGLE_IMAGE_LATENCY * A100_COST_PER_SEC:.4f}")
print(f"Cost per call (multi):  ${MULTI_IMAGE_LATENCY * A100_COST_PER_SEC:.4f}")
print(f"Cost per 1000 buildings (4 img): ${MULTI_IMAGE_LATENCY * 1000 * A100_COST_PER_SEC:.2f}")
print(f"")
print(f"⚠ Endpoint costs ${A100_COST_PER_HOUR:.2f}/hr while deployed.")
print(f"   Scale to 0 when not in use to stop billing.")
print(f"   Cold start penalty: 60-120 seconds.")

## 9. Clean Up (IMPORTANT)

Run this cell to delete the endpoint and stop billing.

In [ ]:
# Option A: Scale to 0 (keeps endpoint definition, stops billing)
print("Scaling endpoint to 0 (stop billing, keep endpoint)...")
endpoint.traffic = {}
client.begin_create_or_update(endpoint).wait()
print(f"✅ Endpoint {ENDPOINT_NAME} scaled to 0")

# Option B: Full delete (uncomment to use)
# print(f"Deleting endpoint {ENDPOINT_NAME}...")
# client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
# print("✅ Endpoint deleted")

## Conclusion

**If all cells ran successfully**:
- ✅ Online endpoint works for VLM inference
- ✅ Single and multi-image inference tested
- ✅ Latency measured for cost estimation
- ✅ Cold start behavior understood

**Next**: Run `02-batch-endpoint.ipynb` to test batch endpoints.
Then return to main plan Phase 02.